# Lesson 21: Ethics & Fairness in Machine Learning

*ML Academy — companion notebook*

Where bias comes from, how to measure fairness per group, why the criteria conflict, and what to do about it.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-21.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 10. In Python: a fairness report from scratch

In [1]:
import numpy as np
import pandas as pd

# A synthetic loan dataset: two groups with different base rates of repayment
rng = np.random.default_rng(0)
n = 2000
group = rng.choice(["A", "B"], size=n, p=[0.6, 0.4])
base = np.where(group == "A", 0.5, 0.2)              # share who would repay
repay = (rng.random(n) < base).astype(int)
score = np.clip(0.5 + 0.25 * (repay - 0.5) * 2 + rng.normal(0, 0.18, n), 0, 1)  # model's score
df = pd.DataFrame({"group": group, "repay": repay, "score": score})

def group_report(df, threshold):
    d = df.assign(approve=(df["score"] >= threshold).astype(int))
    rows = {}
    for g, s in d.groupby("group"):
        tp = ((s.approve == 1) & (s.repay == 1)).sum(); fp = ((s.approve == 1) & (s.repay == 0)).sum()
        fn = ((s.approve == 0) & (s.repay == 1)).sum(); tn = ((s.approve == 0) & (s.repay == 0)).sum()
        rows[g] = {"selection rate": (tp + fp) / len(s), "TPR": tp / (tp + fn), "FPR": fp / (fp + tn), "PPV": tp / (tp + fp)}
    return pd.DataFrame(rows).T.round(3)

r = group_report(df, threshold=0.6)
print(r)
print("demographic-parity ratio:", round(r.loc["B", "selection rate"] / r.loc["A", "selection rate"], 3))

   selection rate    TPR    FPR    PPV
A           0.430  0.799  0.039  0.956
B           0.178  0.801  0.025  0.887
demographic-parity ratio: 0.414


## 13. Going deeper: the impossibility result, measured

In [2]:
import numpy as np

# Two groups with different base rates; a model whose scores are perfectly calibrated in both groups
rng = np.random.default_rng(0)
def make_group(n, a, b):
    risk = rng.beta(a, b, n)                 # each person's true probability
    return risk, (rng.random(n) < risk).astype(int)
groups = {"group A": make_group(20000, 2, 5), "group B": make_group(20000, 4, 4)}

print("group     base rate   calibration (pred 0.6–0.7: actual)   TPR     FPR     PPV")
for g, (score, y) in groups.items():
    flag = score >= 0.5
    sel = (score >= 0.6) & (score < 0.7)
    tpr = flag[y == 1].mean(); fpr = flag[y == 0].mean(); ppv = y[flag].mean()
    print(f"{g:8}   {y.mean():9.3f}   {y[sel].mean():33.3f}   {tpr:.3f}   {fpr:.3f}   {ppv:.3f}")

group     base rate   calibration (pred 0.6–0.7: actual)   TPR     FPR     PPV
group A        0.287                               0.629   0.232   0.060   0.607
group B        0.500                               0.654   0.631   0.362   0.636


In [3]:
# Equalise true-positive rates with group-specific thresholds, and see what it costs
def rates(score, y, t):
    flag = score >= t
    return flag[y == 1].mean(), flag[y == 0].mean(), (flag == y).mean()

sA, yA = groups["group A"]; sB, yB = groups["group B"]
target_tpr = rates(sB, yB, 0.5)[0]
ts = np.linspace(0.05, 0.95, 181)
tA = ts[np.argmin([abs(rates(sA, yA, t)[0] - target_tpr) for t in ts])]
for name, t_A in [("one threshold 0.5 for both", 0.5), (f"threshold {tA:.3f} for group A", tA)]:
    tprA, fprA, accA = rates(sA, yA, t_A); tprB, fprB, accB = rates(sB, yB, 0.5)
    print(f"{name:28}: TPR A {tprA:.3f} / B {tprB:.3f}   FPR A {fprA:.3f} / B {fprB:.3f}   accuracy A {accA:.3f}")

one threshold 0.5 for both  : TPR A 0.232 / B 0.631   FPR A 0.060 / B 0.362   accuracy A 0.736
threshold 0.305 for group A : TPR A 0.634 / B 0.631   FPR A 0.320 / B 0.362   accuracy A 0.667


In [4]:
# Small groups make fairness metrics noisy: bootstrap interval for a TPR measured on just 400 people
yS, sS = yA[:400], sA[:400]
pos = np.flatnonzero(yS == 1)
boot = [np.mean(sS[rng.choice(pos, len(pos))] >= 0.5) for _ in range(5000)]
print("positives in the small sample:", len(pos), "  TPR:", round(np.mean(sS[pos] >= 0.5), 3),
      "  95% interval:", np.round(np.percentile(boot, [2.5, 97.5]), 3))

positives in the small sample: 106   TPR: 0.226   95% interval: [0.151 0.311]


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict
cols = ["dist_km", "age_yrs"]                 # the model never sees "area"
clf = make_pipeline(StandardScaler(), LogisticRegression())
df["pred"] = cross_val_predict(clf, df[cols], df["sold_fast"], cv=4)   # honest out-of-fold predictions
df["group"] = np.where(df["area"] == "Central", "Central", "Not central")
for g, s in df.groupby("group"):
    tp = ((s.pred == 1) & (s.sold_fast == 1)).sum(); fp = ((s.pred == 1) & (s.sold_fast == 0)).sum()
    pos, neg = (s.sold_fast == 1).sum(), (s.sold_fast == 0).sum()
    print(f"{g:12s} n={len(s):2d}  base rate {s.sold_fast.mean():.3f}  flagged {s.pred.mean():.3f}  "
          f"TPR {tp / pos:.2f}  FPR {fp / neg if neg else float('nan'):.2f}")

Central      n= 4  base rate 0.750  flagged 1.000  TPR 1.00  FPR 1.00
Not central  n= 8  base rate 0.375  flagged 0.375  TPR 1.00  FPR 0.00


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [6]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Selection rate per group

Compute the share of applicants approved in each group.

In [7]:
import pandas as pd
df = pd.DataFrame({"group": ["A", "A", "A", "A", "B", "B", "B", "B"], "approved": [1, 1, 0, 1, 0, 1, 0, 0]})
rates = df["approved"]  # ← replace with the per-group mean
ratio = round(rates.min() / rates.max(), 3)
check(ratio, 0.333, hint='Group by the group column and take the mean of the 0/1 approval column.')

❌ Not yet. Hint: Group by the group column and take the mean of the 0/1 approval column.


### Challenge 2: True-positive rate for one group

Compute the TPR: approved repayers divided by all repayers.

In [8]:
import numpy as np
repay = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
approve = np.array([1, 1, 1, 0, 1, 0, 0, 0, 0, 0])
tpr = approve.mean()  # ← replace: only count people who would repay
check(tpr, 0.75, hint='Look only at people with repay == 1 and take the share approved.')

❌ Not yet. Hint: Look only at people with repay == 1 and take the share approved.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Selection rate per group** — the missing line:

```python
rates = df.groupby("group")["approved"].mean()
```

**2. True-positive rate for one group** — the missing line:

```python
tpr = (approve[repay == 1] == 1).mean()
```